In [1]:
import pandas as pd

df_analysis_deer_harvest = pd.read_csv("df_combined_harvest.csv")
df_analysis_deer_harvest.columns

Index(['County', 'Antlered_2014-15', 'Antlered_2015-16', 'Antlerless_2014-15',
       'Antlerless_2015-16', 'Total_2014-15', 'Total_2015-16',
       'Antlered_2016-17', 'Antlerless_2016-17', 'Total_2016-17',
       'Antlered_2016-17.1', 'Antlered_2017-18', 'Antlerless_2016-17.1',
       'Antlerless_2017-18', 'Total_2016-17.1', 'Total_2017-18',
       'Antlered_2017-18.1', 'Antlered_2018-19', 'Antlerless_2017-18.1',
       'Antlerless_2018-19', 'Total_2017-18.1', 'Total_2018-19',
       'Antlered_2019-20', 'Antlerless_2019-20', 'Total_2019-20',
       'Antlered_2020-21', 'Antlerless_2020-21', 'Total_2020-21',
       'Antlered_2021-22', 'Antlerless_2021-22', 'Total_2021-22',
       'Antlered_2022-23', 'Antlerless_2022-23', 'Total_2022-23',
       'Antlered_2023-24', 'Antlerless_2023-24', 'Total_2023-24',
       'Antlered_2024-25', 'Antlerless_2024-25', 'Total_2024-25',
       'Antlered_2025-26', 'Antlerless_2025-26', 'Total_2025-26'],
      dtype='str')

In [2]:
#Make wide concat table long using melt and keep county as unit of observation  

df_long = df_analysis_deer_harvest.melt(id_vars='County')

In [3]:
#Separate antlered and antlerless classifications from their years, creating separate columns for those classifications and the hunting seasons
#https://stackoverflow.com/questions/14745022/how-to-split-a-dataframe-string-column-into-two-columns

df_long[['Type', 'Season']] = df_long['variable'].str.rsplit('_', n=1, expand=True)

In [4]:
#Hunting season spreads across two years (typically begins in fall and carries into winter of following year
#But just use the start year in the table  values 

#df_long['Season'] = df_long['Season'].str[:4].astype(int)
#^from ChatGPT 

df_long['Season'] = df_long['Season'].str[:4]

In [5]:
#Drop variable column now that it's been split into the columns I want 
df_long = df_long.drop(columns = "variable")

In [6]:
df_long=df_long.rename(columns={"value": "Deer_Harvest_Count", "Season": "Hunting_Season"})

In [7]:
#Remove a bunch of NaNs bc concat for 10 separate tables created a bunch of cells with no values

df_long=df_long.dropna(subset=["Deer_Harvest_Count"])
#https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.dropna.html

In [8]:
df_long

,County,Deer_Harvest_Count,Type,Hunting_Season
0,Allegany,1731.0,Antlered,2014
1,Anne Arundel,817.0,Antlered,2014
2,Baltimore,1502.0,Antlered,2014
3,Calvert,470.0,Antlered,2014
4,Caroline,735.0,Antlered,2014
...,...,...,...,...
10621,Somerset,1846.0,Total,2025
10622,Talbot,1951.0,Total,2025
10623,Washington,4560.0,Total,2025
10624,Wicomico,2787.0,Total,2025


In [9]:
#"Total" index label initially at the bottom of each county list with its own row of values. Moved under "Type" when variable column was split. 
#Remove "Total" rows/values from combined table 


#https://www.geeksforgeeks.org/pandas/drop-rows-from-the-dataframe-based-on-certain-condition-applied-on-a-column/
indices_to_drop = df_long[df_long['Type'] == "Total"].index

In [10]:
df_long.drop(indices_to_drop, inplace=True)

In [11]:
#Later noticed, when creating visuals I think, that Saint Mary's had two different spellings

df_long = df_long.replace({"County": {"Saint Mary’s": "St. Mary's"}})

In [12]:
#Also later noticed that St. Mary's used two different apostrophes 

df_long = df_long.replace({"County": {"St. Mary's": "St. Mary’s"}})

In [13]:
#Double check that only 23 names are in county, as there are only 23 counties 

df_long['County'].unique()

<ArrowStringArray>
[       'Allegany',    'Anne Arundel',       'Baltimore',         'Calvert',
        'Caroline',         'Carroll',           'Cecil',         'Charles',
      'Dorchester',       'Frederick',         'Garrett',         'Harford',
          'Howard',            'Kent',      'Montgomery', 'Prince George’s',
    'Queen Anne’s',      'St. Mary’s',        'Somerset',          'Talbot',
      'Washington',        'Wicomico',       'Worcester']
Length: 23, dtype: str

In [14]:
df_long.query("County == 'St. Mary’s' and Hunting_Season == '2018-19'")

,County,Deer_Harvest_Count,Type,Hunting_Season


In [15]:
##------Next several cells: finding and dropping duplicate values-------##

#Noticed that values for a few years, when creating visuals, had repeated. 
#Original tables that were concacted repeated previous year as well reporting year, so all years were duplicated in the table
#Additionally, some county totals were reported twice because their numbers changed by 1 across the different annual tables 

In [16]:
df_long = df_long.drop_duplicates()

In [17]:
df_long.groupby("Hunting_Season")["Deer_Harvest_Count"].sum()

Hunting_Season
2014    86883.0
2015    84022.0
2016    85193.0
2017    86542.0
2018    77382.0
2019    83152.0
2020    83303.0
2021    70845.0
2022    76687.0
2023    72642.0
2024    84201.0
2025    71649.0
Name: Deer_Harvest_Count, dtype: float64

In [18]:
df_long["Hunting_Season"].value_counts()

Hunting_Season
2019    48
2020    47
2014    46
2015    46
2016    46
2017    46
2018    46
2021    46
2022    46
2023    46
2024    46
2025    46
Name: count, dtype: int64

In [19]:
df_long.query("Hunting_Season == '2019-20'")

,County,Deer_Harvest_Count,Type,Hunting_Season


In [20]:
df_long=df_long.drop(index=[5687])

In [21]:
df_long.reset_index(drop=True)

,County,Deer_Harvest_Count,Type,Hunting_Season
0,Allegany,1731.0,Antlered,2014
1,Anne Arundel,817.0,Antlered,2014
2,Baltimore,1502.0,Antlered,2014
3,Calvert,470.0,Antlered,2014
4,Caroline,735.0,Antlered,2014
...,...,...,...,...
549,Somerset,1189.0,Antlerless,2025
550,Talbot,1238.0,Antlerless,2025
551,Washington,2223.0,Antlerless,2025
552,Wicomico,1675.0,Antlerless,2025


In [22]:
df_long["Hunting_Season"].value_counts()

Hunting_Season
2019    47
2020    47
2014    46
2015    46
2016    46
2017    46
2018    46
2021    46
2022    46
2023    46
2024    46
2025    46
Name: count, dtype: int64

In [23]:
df_long.query("Hunting_Season == '2019-20'")

,County,Deer_Harvest_Count,Type,Hunting_Season


In [24]:
df_long=df_long.drop(index=[5434])

In [25]:
df_long.reset_index(drop=True)

,County,Deer_Harvest_Count,Type,Hunting_Season
0,Allegany,1731.0,Antlered,2014
1,Anne Arundel,817.0,Antlered,2014
2,Baltimore,1502.0,Antlered,2014
3,Calvert,470.0,Antlered,2014
4,Caroline,735.0,Antlered,2014
...,...,...,...,...
548,Somerset,1189.0,Antlerless,2025
549,Talbot,1238.0,Antlerless,2025
550,Washington,2223.0,Antlerless,2025
551,Wicomico,1675.0,Antlerless,2025


In [26]:
df_long["Hunting_Season"].value_counts()

Hunting_Season
2020    47
2014    46
2015    46
2016    46
2017    46
2018    46
2019    46
2021    46
2022    46
2023    46
2024    46
2025    46
Name: count, dtype: int64

In [27]:
df_long.query("Hunting_Season == '2020-21'")

,County,Deer_Harvest_Count,Type,Hunting_Season


In [28]:
df_long=df_long.drop(index=[6481])

In [29]:
df_long.reset_index(drop=True)

,County,Deer_Harvest_Count,Type,Hunting_Season
0,Allegany,1731.0,Antlered,2014
1,Anne Arundel,817.0,Antlered,2014
2,Baltimore,1502.0,Antlered,2014
3,Calvert,470.0,Antlered,2014
4,Caroline,735.0,Antlered,2014
...,...,...,...,...
547,Somerset,1189.0,Antlerless,2025
548,Talbot,1238.0,Antlerless,2025
549,Washington,2223.0,Antlerless,2025
550,Wicomico,1675.0,Antlerless,2025


In [30]:
df_long["Hunting_Season"].value_counts()

Hunting_Season
2014    46
2015    46
2016    46
2017    46
2018    46
2019    46
2020    46
2021    46
2022    46
2023    46
2024    46
2025    46
Name: count, dtype: int64

In [36]:
#Create a new column called "Hunting_Region" in the combined table that can be used to extract values based on the two different hunting regions in the state:
    #Region A counties
    #Region B counties
    #Suburban Management zone 

#https://www.eregulations.com/maryland/hunting/deer-seasons-bag-limits

#https://www.geeksforgeeks.org/pandas/adding-new-column-to-existing-dataframe-in-pandas/
# address dictionary with names as keys & addresses as values
hunting_region = {'Allegany': 'Region A', 
           'Anne Arundel': 'Region B', 
           'Baltimore': 'Region B', 
           'Calvert': 'Region B',
           'Caroline': 'Region B',
           'Carroll': 'Region B',
           'Cecil': 'Region B', 
           'Charles': 'Region B', 
           'Dorchester': 'Region B', 
           'Frederick': 'Region B',
           'Garrett': 'Region A',
           'Harford': 'Region B',
           'Howard': 'Region B', 
           'Kent': 'Region B', 
           'Montgomery': 'Region B',
           'Prince George’s': 'Region B',
           'Queen Anne’s': 'Region B',
           'St. Mary’s': 'Region B',
           'Somerset': 'Region B',
           'Talbot': 'Region B', 
           'Washington': 'Mixed', 
           'Wicomico': 'Region B',
           'Worcester': 'Region B'
          }
df_long['Hunting_Region'] = df_long['County'].map(hunting_region)

In [37]:
#Suburban_management_zone
#Because suburban management zone is in Region B, create separate column with boolean values 

df_long["Suburban_Management_Zone"] = df_long["County"].isin([
    "Baltimore", 
    "Howard",
    "Montgomery",
    "Anne Arundel",
    "Prince George’s"])

In [42]:
#Total reported deer harvest counts by hunting season across Maryland 

df_harvest_over_time = df_long.groupby(["Hunting_Season", "Type", "Hunting_Region", "Suburban_Management_Zone"], as_index=False)["Deer_Harvest_Count"].sum()

In [43]:
df_harvest_over_time.to_csv("df_harvest_over_time.csv")

In [44]:
df_harvest_over_time

,Hunting_Season,Type,Hunting_Region,Suburban_Management_Zone,Deer_Harvest_Count
0,2014,Antlered,Mixed,False,2026.0
1,2014,Antlered,Region A,False,4217.0
2,2014,Antlered,Region B,False,16697.0
3,2014,Antlered,Region B,True,5341.0
4,2014,Antlerless,Mixed,False,3061.0
...,...,...,...,...,...
91,2025,Antlered,Region B,True,5119.0
92,2025,Antlerless,Mixed,False,2223.0
93,2025,Antlerless,Region A,False,3273.0
94,2025,Antlerless,Region B,False,27211.0


In [46]:
#REGION A harvest over time 
#Mixed = Washington County 

regionA_df_harvest_over_time = df_long[df_long["Hunting_Region"].isin(["Region A","Mixed"])].groupby(["County","Hunting_Season", "Type"], as_index=False)["Deer_Harvest_Count"].sum()
#^with help from ChatGPT 

In [48]:
regionA_df_harvest_over_time.to_csv("regionA_df_harvest_over_time.csv")

In [49]:
#REGION B harvest over time 

regionB_df_harvest_over_time = df_long[df_long["Hunting_Region"].isin(["Region B","Mixed"])].groupby(["County","Hunting_Season", "Type"], as_index=False)["Deer_Harvest_Count"].sum()


In [50]:
regionB_df_harvest_over_time.to_csv("regionB_df_harvest_over_time.csv")

In [51]:
#Total harvest over time for MD suburban management zone before and after it was created 

#“A suburban deer management zone comprised of Anne Arundel, Baltimore, Howard, Montgomery, and Prince George’s counties 
#has been established. This zone has an unlimited antlerless deer bag limit for archery season" (18). 
#https://dnr.maryland.gov/wildlife/Documents/MD-Annual-Deer-Report-2019-2020.pdf

#Before creation of zone 
df_before_zone = df_long[
    (df_long["Hunting_Season"] < "2020-21") &
    (df_long["County"].isin(["Anne Arundel", "Baltimore", "Howard", "Montgomery", "Prince George’s"]))]

df_before_zone

,County,Deer_Harvest_Count,Type,Hunting_Season,Hunting_Region,Suburban_Management_Zone
1,Anne Arundel,817.0,Antlered,2014,Region B,True
2,Baltimore,1502.0,Antlered,2014,Region B,True
12,Howard,682.0,Antlered,2014,Region B,True
14,Montgomery,1520.0,Antlered,2014,Region B,True
15,Prince George’s,820.0,Antlered,2014,Region B,True
...,...,...,...,...,...,...
6441,Anne Arundel,1277.0,Antlerless,2020,Region B,True
6442,Baltimore,3575.0,Antlerless,2020,Region B,True
6452,Howard,1576.0,Antlerless,2020,Region B,True
6454,Montgomery,2741.0,Antlerless,2020,Region B,True


In [52]:
df_before_zone_harvest_over_time = df_before_zone.groupby(["County", "Hunting_Season", "Type", "Hunting_Region", "Suburban_Management_Zone"], as_index=False)["Deer_Harvest_Count"].sum()

In [54]:
df_before_zone_harvest_over_time.to_csv("df_before_zone_harvest_over_time.csv")

In [55]:
#After creation of zone 
df_after_zone = df_long[
    (df_long["Hunting_Season"] >= "2020-21") &
    (df_long["County"].isin(["Anne Arundel", "Baltimore", "Howard", "Montgomery", "Prince George’s"]))]


In [56]:
df_after_zone_harvest_over_time = df_after_zone.groupby(["County", "Hunting_Season", "Type", "Hunting_Region", "Suburban_Management_Zone"], as_index=False)["Deer_Harvest_Count"].sum()


In [57]:
df_after_zone_harvest_over_time.to_csv("df_after_zone_harvest_over_time.csv")

In [58]:
#Needed to reshape data in order to create stacked bar chart to show anterless to antlered ratios 

regionA_stacked_df_harvest_over_time = regionA_df_harvest_over_time.pivot(index = ["Hunting_Season", "County"],
                                                          columns = ["Type"], 
                                                          values = "Deer_Harvest_Count")     
#^help from ChatGPT

In [59]:
regionA_stacked_df_harvest_over_time
regionA_stacked_df_harvest_over_time.to_csv("stacked_df_harvest_over_time.csv")

In [60]:
regionB_stacked_df_harvest_over_time = regionB_df_harvest_over_time.pivot(index = ["Hunting_Season", "County"],
                                                          columns = ["Type"], 
                                                          values = "Deer_Harvest_Count")    

In [61]:
##------Next several cells: Created more DFs for tables that didn't end up in the story-------##

In [57]:
eastern_regionB_df_harvest_over_time = regionB_df_harvest_over_time[
    regionB_df_harvest_over_time["County"].isin (["Caroline", "Cecil", "Dorchester", "Kent", "Queen Anne’s", 
                                                  "Somerset", "Talbot", "Wicomico", "Worcester"])]

In [58]:
eastern_regionB_df_harvest_over_time_stacked = eastern_regionB_df_harvest_over_time.pivot(index = ["Hunting_Season", "County"],
                                                          columns = ["Type"], 
                                                          values = "Deer_Harvest_Count")  

In [59]:
eastern_regionB_df_harvest_over_time_stacked.to_csv("eastern_regionB_df_harvest_over_time_stacked.csv")

In [60]:
regionB_stacked_df_harvest_over_time.to_csv("regionB_stacked_df_harvest_over_time.csv")

In [62]:
#Instead of creating charts that showed each county, decided to combine county totals by region to show any changes from 2014-2015 in harvests of antlerless to antlered deer

df_before_zone_stacked_harvest_over_time = df_before_zone_harvest_over_time.pivot(index = ["Hunting_Season", "County"],
                                                          columns = ["Type"], 
                                                          values = "Deer_Harvest_Count")

In [63]:
#Percentage of antlerless to total harvested deer
total=df_before_zone_stacked_harvest_over_time["Antlered"] + df_before_zone_stacked_harvest_over_time["Antlerless"]
total

# df_before_zone_stacked_harvest_over_time[Percentage_Antlerless_to_Total] = 

Hunting_Season  County         
2014            Anne Arundel       2892.0
                Baltimore          5413.0
                Howard             2581.0
                Montgomery         5410.0
                Prince George’s    2668.0
2015            Anne Arundel       2692.0
                Baltimore          4970.0
                Howard             2334.0
                Montgomery         4744.0
                Prince George’s    2512.0
2016            Anne Arundel       2790.0
                Baltimore          5367.0
                Howard             2400.0
                Montgomery         4873.0
                Prince George’s    2477.0
2017            Anne Arundel       2979.0
                Baltimore          5559.0
                Howard             2427.0
                Montgomery         4783.0
                Prince George’s    2388.0
2018            Anne Arundel       2482.0
                Baltimore          4604.0
                Howard             2027.0
  

In [64]:
df_before_zone_stacked_harvest_over_time['Antlerless_PCT'] = (df_before_zone_stacked_harvest_over_time['Antlerless']/total)*100

In [65]:
df_before_zone_stacked_harvest_over_time.to_csv("df_before_zone_stacked_harvest_over_time.csv")

In [66]:
df_after_zone_stacked_harvest_over_time = df_after_zone_harvest_over_time.pivot(index = ["Hunting_Season", "County"],
                                                          columns = ["Type"], 
                                                          values = "Deer_Harvest_Count")

In [67]:
#Percentage of antlerless to total harvested deer
total_two=df_after_zone_stacked_harvest_over_time["Antlered"] + df_after_zone_stacked_harvest_over_time["Antlerless"]
total_two


Hunting_Season  County         
2021            Anne Arundel       1837.0
                Baltimore          4501.0
                Howard             2019.0
                Montgomery         3306.0
                Prince George’s    1630.0
2022            Anne Arundel       1883.0
                Baltimore          4846.0
                Howard             2109.0
                Montgomery         3433.0
                Prince George’s    1647.0
2023            Anne Arundel       1724.0
                Baltimore          4550.0
                Howard             2054.0
                Montgomery         3362.0
                Prince George’s    1427.0
2024            Anne Arundel       2269.0
                Baltimore          5239.0
                Howard             2181.0
                Montgomery         3464.0
                Prince George’s    1806.0
2025            Anne Arundel       1422.0
                Baltimore          4779.0
                Howard             2105.0
  

In [68]:
df_after_zone_stacked_harvest_over_time['Antlerless_PCT'] = (df_after_zone_stacked_harvest_over_time['Antlerless']/total_two)*100

In [69]:
df_after_zone_stacked_harvest_over_time.to_csv("df_after_zone_stacked_harvest_over_time.csv")

In [70]:
combined_sum_regionA = regionA_stacked_df_harvest_over_time.groupby(level="Hunting_Season").sum()

In [72]:
combined_sum_regionA.to_csv("combined_sum_regionA.csv")

In [73]:
combined_sum_regionB = regionB_stacked_df_harvest_over_time.groupby(level="Hunting_Season").sum()

In [74]:
combined_sum_regionB.to_csv("combined_sum_regionB.csv")